# DeepConvLSTM Reproduction — OPPORTUNITY

Colab notebook for documenting a reproduction attempt using the modern PyTorch implementation. This is **not yet a verified exact reproduction** of the original paper.

## Before training
The upstream implementation and this reproduction may differ from the paper in preprocessing, subject/session split, and optimizer settings. Inspect these choices before interpreting results. A classifier output-size bug was identified in the previous run: `SensorDataset.n_classes` was set to the number of windows rather than the number of activity classes. Do not treat metrics from that run as valid.

## 1. Get the upstream implementation
This notebook clones the upstream project; the repository source files are not duplicated in this notebook.

In [ ]:
!git clone https://github.com/STRCWearlab/DeepConvLSTM_py3.git
%cd DeepConvLSTM_py3
!ls

## 2. Preprocess OPPORTUNITY
Place the dataset where the upstream preprocessing script expects it, then run preprocessing. The dataset itself is intentionally not stored in this GitHub repository.

In [ ]:
!python preprocess_opportunity.py
!find data/opportunity -maxdepth 1 -type f

## 3. Check the generated windows and labels
Expected data arrays have shape `(number_of_windows, 24, 113)` in this configuration. Confirm the actual shapes and class labels for your run.

In [ ]:
import numpy as np

for split in ("train", "val", "test"):
    path = f"data/opportunity/{split}_data.npz"
    dataset = np.load(path)
    x, y = dataset["data"], dataset["target"]
    print(f"{split}: data={x.shape}, target={y.shape}")
    print("  unique labels:", np.unique(y))
    print("  number of classes:", len(np.unique(y)))

## 4. Inspect the class-count definition before training
In the earlier run, `datasets.py` defined `self.n_classes = self.target.shape[0]`, which counts windows rather than classes. Correct this before training so the classifier has one output per activity class. Check the upstream code and label mapping rather than assuming the class count from dataset length.

In [ ]:
!grep -n -E "n_classes|classifier|class SensorDataset" datasets.py DeepConvLSTM_py3.py

## 5. Training (only after checking the class count)
The upstream script may train for a long time and create large checkpoint files. Run it only after correcting and verifying the class-count issue. Store checkpoints in Google Drive or another external location; do not commit them to Git.

In [ ]:
# Run only after verifying/fixing the class-count definition in datasets.py
# !python DeepConvLSTM_py3.py

## Reproducibility notes
- Keep dataset files, generated results, and model checkpoints out of version control.
- Record the exact code revision, preprocessing options, data split, random seeds, optimizer, learning-rate schedule, and class mapping for each run.
- Do not report earlier metrics as final results until the classifier output size is fixed and the model is retrained and evaluated.